# Steel Quality Prediction - Sheyda Asadi

Welcome to my notebook for the **Steel Quality Prediction** competition on Kaggle.  
This competition was hosted by **Mohammad Saeid**

Let's get started ...


Steel Quality Prediction - Sheyda Asadi¶
Welcome to my notebook for the Steel Quality Prediction competition on Kaggle.
This competition was hosted by Mohammad Saeid

Let's get started ...

In this competition, the goal is to predict the quality score of a steel product based on various production parameters. These include temperature, cooling rate, luminosity, and other machine/process-related features. Being able to predict the quality score can help optimize the manufacturing process and reduce defects.The dataset provides both numerical and categorical features, along with a target column (quality_score) for training.

Since the target variable (quality_score) is a continuous number, this is a regression problem. That means our model will try to predict a numerical value. The evaluation metric for this competition is RMSE, which measures how far the predictions are from the true values.

-(I tried RandomForest early on but it wasn’t working well because it was too slow & overfitting. Also played with feature combinations – some of them helped, some didn’t. Didn’t use PCA or anything fancy – just stuck with basic tabular tricks.)-

In this competition, the goal is to predict the quality score of a steel product based on various production parameters. These include temperature, cooling rate, luminosity, and other machine/process-related features. Being able to predict the quality score can help optimize the manufacturing process and reduce defects.The dataset provides both numerical and categorical features, along with a target column (quality_score) for training.

Since the target variable (quality_score) is a continuous number, this is a regression problem. That means our model will try to predict a numerical value. The evaluation metric for this competition is RMSE, which measures how far the predictions are from the true values.

-(I tried RandomForest early on but it wasn’t working well because it was too slow & overfitting.
Also played with feature combinations – some of them helped, some didn’t.
Didn’t use PCA or anything fancy – just stuck with basic tabular tricks.)-



In [ ]:
# If you're running this on Colab, you'll probably need to reinstall these to avoid version issues
# !pip install numpy==1.23.5 pandas lightgbm xgboost catboost scikit-learn optuna --force-reinstall --no-cache-dir
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb
import optuna

df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_test['quality_score'] = np.nan
combined = pd.concat([df_train, df_test], ignore_index=True)


Feature engineering helps models learn better by creating new features from the existing ones.I extracted the dayofweek and is_weekend from the production_date.I also split production_time into hour and minute.Created new features by combining existing ones like temperature / processing_time, defect_area * min_luminosity....These kinds of features can reveal hidden relationships that make the model stronger.

Feature engineering helps models learn better by creating new features from the existing ones.I extracted the dayofweek and is_weekend from the production_date.I also split production_time into hour and minute.Created new features by combining existing ones like temperature / processing_time, defect_area * min_luminosity....These kinds of features can reveal hidden relationships that make the model stronger.

In [ ]:
#Let’s pull out some time-based features – weekdays, weekends.... and Breaking down time into hour and minute


def engineer_features(data):
    data['production_date'] = pd.to_datetime(data['production_date'])
    data['week_day'] = data['production_date'].dt.dayofweek
    data['weekend_flag'] = (data['week_day'] >= 5).astype(int)

    if 'production_time' in data.columns:
        time_split = data['production_time'].str.split(':', expand=True).astype(int)
        data['hour'] = time_split[0]
        data['minute'] = time_split[1]
        data.drop(columns=['production_time'], inplace=True)

    data.drop(columns=['production_date'], inplace=True)

    data['area_x_lum'] = data['defect_area'] * data['min_luminosity']
    data['temp_time_ratio'] = data['temperature'] / (data['processing_time'] + 1e-3)
    data['cooling_time_ratio'] = data['cooling_rate'] / (data['processing_time'] + 1e-3)
    data['luminosity_per_temp'] = data['total_luminosity'] / (data['temperature'] + 1e-3)
    data['temp_mul_cooling'] = data['temperature'] * data['cooling_rate']
    data['lum_mul_area'] = data['total_luminosity'] * data['defect_area']
    return data

combined = engineer_features(combined)

Machine learning models can't directly work with text labels. So, Id Label Encoding to convert categorical features like machine_id and operator_id into numbers.


Machine learning models can't directly work with text labels. So, Id Label Encoding to convert categorical features like machine_id and operator_id into numbers.

In [ ]:
categorical_cols = ['machine_id', 'operator_id']
for col in categorical_cols:
    combined[col] = LabelEncoder().fit_transform(combined[col])

To make sure the model generalizes well, I used 5-fold cross validation.In each fold, I trained LightGBM, CatBoost, and XGBoost models.Then saved their predictions.Finally,trained the Ridge meta model using those predictions.This process helps blend the strengths of all three models

To make sure the model generalizes well, I used 5-fold cross validation.In each fold, I trained LightGBM, CatBoost, and XGBoost models.Then saved their predictions.Finally,trained the Ridge meta model using those predictions.This process helps blend the strengths of all three models

In [ ]:
train_set = combined[~combined['quality_score'].isna()].copy()
test_set = combined[combined['quality_score'].isna()].copy()
X_train = train_set.drop(columns=['id', 'quality_score'])
y_train = train_set['quality_score']
X_test = test_set.drop(columns=['id', 'quality_score'])
folds = KFold(n_splits=5, shuffle=True, random_state=2024)


Instead of manually trying different parameters, I used Optuna to automatically find the best settings for the LightGBM model. It runs multiple trials and picks the combination that gives the best score.This helps make the model more accurate

Instead of manually trying different parameters, I used Optuna to automatically find the best settings for the LightGBM model. It runs multiple trials and picks the combination that gives the best score.This helps make the model more accurate

In [ ]:
# Using Optuna to tune some LGB params – saves time compared to GridSearch


def optuna_objective(trial):
    config = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 2024
    }
    rmse_scores = []
    for train_idx, val_idx in folds.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = lgb.LGBMRegressor(**config)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmse_scores.append(rmse)
    return np.mean(rmse_scores)
    
# Running optimization – might take a bit depending on your machine
study = optuna.create_study(direction='minimize')
study.optimize(optuna_objective, n_trials=30)

# Tweak the n_estimators a bit manually (worked better in my tests)
optimal_params = study.best_params
optimal_params['n_estimators'] = 1200
optimal_params['random_state'] = 2024

LightGBM is fast and works well with large data.CatBoost is good at handling categorical features automatically.XGBoost is a classic gradient boosting model with solid performance. By using all three, we can benefit from their strengths and hopefully get better overall performance. and I used RidgeCV as the meta-model to combine predictions from LightGBM,CatBoost,and XGBoost because It helps improve accuracy by combining different models,Reduces the risk of overfitting,It can learn from patterns that each individual model might miss.

LightGBM is fast and works well with large data.CatBoost is good at handling categorical features automatically.XGBoost is a classic gradient boosting model with solid performance.
By using all three, we can benefit from their strengths and hopefully get better overall performance.
and I used RidgeCV as the meta-model to combine predictions from LightGBM,CatBoost,and XGBoost because It helps improve accuracy by combining different models,Reduces the risk of overfitting,It can learn from patterns that each individual model might miss.

In [ ]:
val_preds = np.zeros((len(X_train), 3))
final_test_preds = np.zeros((len(X_test), 3))

for fold_idx, (train_idx, val_idx) in enumerate(folds.split(X_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    #LightGBM
    lgb_model = lgb.LGBMRegressor(**optimal_params)
    lgb_model.fit(X_tr, y_tr)
    val_preds[val_idx, 0] = lgb_model.predict(X_val)
    final_test_preds[:, 0] += lgb_model.predict(X_test) / folds.n_splits

    #CatBoost
    cb_model = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=6, verbose=0)
    cb_model.fit(X_tr, y_tr, cat_features=[X_train.columns.get_loc(col) for col in categorical_cols])
    val_preds[val_idx, 1] = cb_model.predict(X_val)
    final_test_preds[:, 1] += cb_model.predict(X_test) / folds.n_splits

    #XGBoost
    xgb_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=2024)
    xgb_model.fit(X_tr, y_tr)
    val_preds[val_idx, 2] = xgb_model.predict(X_val)
    final_test_preds[:, 2] += xgb_model.predict(X_test) / folds.n_splits

stacker = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 2, 50)))
stacker.fit(val_preds, y_train)
ensemble_output = stacker.predict(final_test_preds)

In [ ]:
# Creating the submission file – fingers crossed 🤞
submission_file = pd.DataFrame({"id": df_test["id"], "quality_score": ensemble_output})
submission_file.to_csv("submission.csv", index=False)
print("submission.csv")
